# fxhash — file-type composition of each project (interactive)

For every project, how many files of each type (`.js`, `.png`, `.html`, ...) it
contains, as a **stacked bar per project across all projects**. The layers are in
the same order for every project.

The charts are **interactive (Plotly)**:
- **drag** a rectangle to zoom into any range of projects
- **hover** a bar to see the project name and the file count of that type
- **double-click** to reset the zoom

Two views: projects **by release date**, and **alphabetically**. Each is also
saved as a standalone `.html` in `charts/` (plotly.js embedded, so it opens and
zooms in any browser, even offline).

Requires the `projects/` folder (created by `download_projects.py`).

In [ ]:
import os
from collections import Counter

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

PROJECTS = 'projects'
CATEGORIES = ['js', 'html', 'css', 'json', 'png', 'jpg', 'webp', 'svg', 'other']
COLORS = ['#0066ff', '#00a3cc', '#33cc99', '#ffcc00', '#ff8800',
          '#ff3366', '#cc33ff', '#9966ff', '#999999']

def read_source(path):
    name = date = None
    for line in open(path, encoding='utf-8', errors='replace'):
        if line.startswith('name:'):
            name = line.split(':', 1)[1].strip()
        elif line.startswith('mint_opens_at:'):
            date = line.split(':', 1)[1].strip()
    return name, date

rows = []
for root, _, files in os.walk(PROJECTS):
    if '_source.txt' not in files:
        continue
    name, date = read_source(os.path.join(root, '_source.txt'))
    counts = Counter()
    for r, _, fs in os.walk(root):
        if '__MACOSX' in r.split(os.sep):
            continue
        for f in fs:
            if f.startswith('._') or f == '_source.txt':
                continue
            ext = os.path.splitext(f)[1].lower().lstrip('.')
            counts[ext if ext in CATEGORIES else 'other'] += 1
    row = {'name': name or os.path.basename(root), 'date': date}
    for c in CATEGORIES:
        row[c] = counts.get(c, 0)
    rows.append(row)

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')
df['total'] = df[CATEGORIES].sum(axis=1)
df.to_csv('data/project_file_types.csv', index=False)

print(f'{len(df)} projects')
print(df[CATEGORIES].sum().sort_values(ascending=False).to_string())

In [ ]:
os.makedirs('charts', exist_ok=True)

def stack_fig(frame, title, xlabel, html_name):
    frame = frame.reset_index(drop=True)
    x = list(range(1, len(frame) + 1))
    cap = int(df['total'].quantile(0.95))

    fig = go.Figure()
    for cat, color in zip(CATEGORIES, COLORS):
        fig.add_bar(
            x=x, y=frame[cat], name=cat, marker_color=color,
            customdata=frame['name'],
            hovertemplate='<b>%{customdata}</b><br>' + cat + ': %{y} files<extra></extra>',
        )
    fig.update_layout(
        barmode='stack', bargap=0, height=600,
        title=title, xaxis_title=xlabel,
        yaxis_title='# files per project (stacked)',
        legend_traceorder='normal',
        hovermode='closest',
    )
    fig.update_yaxes(range=[0, cap])
    fig.write_html(f'charts/{html_name}', include_plotlyjs=True)
    display(HTML(fig.to_html(include_plotlyjs='inline', full_html=False)))

## 1. Projects ordered by release date

In [ ]:
by_date = df.dropna(subset=['date']).sort_values('date')
stack_fig(by_date,
          f'File types per project \u2014 {len(by_date)} projects by date',
          'projects (oldest \u2192 newest)',
          'file_composition_by_date.html')

## 2. Projects in alphabetical order

In [ ]:
by_name = df.sort_values('name', key=lambda s: s.str.lower())
stack_fig(by_name,
          f'File types per project \u2014 {len(by_name)} projects alphabetical',
          'projects (A \u2192 Z)',
          'file_composition_by_name.html')